<a href="https://colab.research.google.com/github/dnhshl/cc-ai/blob/main/frozenlake26.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# KI in der Robotik: Reinforcement Learning Praxis

Dieses Notebook implementiert einen **Q-Learning Agenten**

DN, 26.01.26

Wir nutzen ein einfaches Beispiel: FrozenLake.

> *Winter is here. You and your friends were tossing around a frisbee at the park when you made a wild throw that left the frisbee out in the middle of the lake. The water is mostly frozen, but there are a few holes where the ice has melted. If you step into one of those holes, you'll fall into the freezing water. At this time, there's an international frisbee shortage, so it's absolutely imperative that you navigate across the lake and retrieve the disc. However, the ice is slippery, so you won't always move in the direction you intend.*


<img src="https://zitaoshen.rbind.io/project/rl/1-min-of-reinforcement-learning-q-learning/featured.png?raw=1" alt="FrozenLake" style="width: 400px;"/>

Wir starten in der linken oberen Ecke (S). Unser Ziel ist es, zur unteren rechten Ecke (G) zu gelangen, ohne zwischendurch in ein Loch (H) zu fallen.

Es gibt vier mögliche Aktionen:

0=links, 1=runter, 2=rechts, 3=rauf

Die Stati sind die 16 Felder (0 .. 15).


Jede Bewegung führt dazu, dass sich der Status des Agenten von $s_t$ zu $s_{t+1}$ ändert, wenn er seinen Standort ändert, es sei denn, er versucht, sich in Richtung einer Wand zu bewegen, was dazu führt, dass sich der Status des Agenten nicht ändert (der Agent bewegt sich nicht).
Für das Erreichen des Ziels (G) erhalten wir eine positive Belohnung von „+1“, die je nach Dauer abgezinst wird.
Obwohl es keine negative Belohnung für das Fallen in ein Loch gibt (H), zahlt der Agent dennoch eine Strafe in dem Sinne, dass das Fallen in das Loch die Episode beendet und ihn daher daran hindert, eine Belohnung zu erhalten.
Wir wollen eine Richtlinie $\pi$ lernen, die uns in möglichst wenigen Schritten von unserem Startort (S) zum Ziel (G) führt.

Das Problem ist schwieriger, als es sich für uns in der ersten Betrachtung darstellt:

- **Kenntnis der Zustände und Übergangswahrscheinlichkeiten:** Aus der globalen Sicht von oben könnte der erste Gedanke sein, einen Weg vom Start bis zum Ziel zu planen, genau wie bei einem Labyrinth.
Diese Ansicht wird uns jedoch den Algorithmus-Designern zur Verfügung gestellt, damit wir das vorliegende Problem visualisieren können.
Der Agent, der die Aufgabe lernt, erhält dieses Vorwissen *nicht*; Alles was wir sagen werden ist, dass es 16 Stati und 4 mögliche Aktionen in jedem Status gibt.
Eine passendere Analogie wäre:  Sie stehen mit verbundenen Augen auf einem zugeforenen See. Jedes mal, wenn Sie sich entscheiden, einen Schritt in eine von vier Richtungen zu machen, wird Ihnen Ihr neuer Zustand (Standort) mitgeteilt. Finden Sie das Frisbee, ohne ins Eis einzubrechen, erhalten Sie eine Belohnung, die umso größer ist, je schneller Sie das Frisbee finden.

- **Kenntnis des Ziels (Belohnung):**
Der Agent weiß  *nicht*, was das Ziel ist.
Vielmehr lernen Sie das Ziel, indem es Belohnungen (oder Strafen) gibt, und der Algorithmus aktualisiert seine Richtlinie zur Wahl von Aktionen so, dass er Aktionen mit größerer Wahrscheinlichkeit erneut durchführt, die voraussichtlich zu einer späteren Belohnung führen.
Beachten Sie, dass dies bedeutet, dass ein Agent, wenn er bestimmte Belohnungen nie erhält, nicht weiß, dass sie existieren.

- **Vorkenntnisse in Pfadfindung, Physik, etc.:** Als Mensch bringen Sie, auch wenn Sie diese Aufgabe noch nicht gelöst haben, enorme Vorkenntnisse zu diesem Problem mit.
Sie wissen beispielsweise, dass der kürzeste Weg zu einem Ziel eine Linie ist.
Sie wissen, dass Norden, Süden, Osten und Westen Richtungen sind und dass Sie nach Norden und dann nach Süden zurückkehren, wo Sie bereits waren.
Sie wissen, dass Eis rutschig ist.
Sie wissen, dass eisiges Wasser kalt ist.
Sie wissen, dass es schlecht ist, in eiskaltem Wasser zu sein.
Der Agent weiß von all diesen Dingen nichts; seine anfängliche Richtlinie besteht im Wesentlichen darin, Aktionen vollständig zufällig auszuwählen.
Am Ende des Trainings wird es immer noch nicht wissen, was abstrakte Konzepte wie "Nord/Süd", "kalt" oder "rutschig" bedeuten, aber es wird (hoffentlich) eine gute Politik gelernt haben, die es ihm ermöglicht, das Ziel zu erreichen.


In [ ]:
# 1. Setup
!pip install gymnasium[toy_text] numpy

import gymnasium as gym
import numpy as np
import random



In [ ]:
# Hyperparameter
alpha = 0.1    # Lernrate
gamma = 0.95   # Diskontierungsfaktor

# Epsilon-Decay Strategie (Erweiterung zu Folie 37)
epsilon = 1.0          # Startwert: 100% Exploration (reiner Zufall)
epsilon_min = 0.01      # Mindestwert: Ein bisschen Zufall bleibt immer
epsilon_decay = 0.999  # Zerfallsrate pro Episode

iterations = 10000

is_slippery = False

In [ ]:
# Training
import gymnasium as gym
import numpy as np
import random

env = gym.make('FrozenLake-v1', map_name='4x4', is_slippery=is_slippery , render_mode="rgb_array")
print('Umgebung geladen.')

state_size = env.observation_space.n
action_size = env.action_space.n

# Q-Table frisch auf 0 setzen
q_table = np.zeros([state_size, action_size])

print("Training startet... ")

found_reward = False
for episode in range(iterations):
    state, info = env.reset()
    done = False

    while not done:
        # Epsilon-Greedy Auswahl
        if random.uniform(0, 1) < epsilon:
            action = env.action_space.sample() # Reiner Zufall
        else:
            action = np.argmax(q_table[state]) # Bestes Wissen

        next_state, reward, terminated, truncated, info = env.step(action)
        done = terminated or truncated

        # Debug-Meldung beim allerersten Erfolg
        if reward > 0 and not found_reward:
            print(f"ERFOLG! In Episode {episode} wurde das Ziel zum ersten Mal gefunden!")
            found_reward = True

        # Die Update-Regel
        old_q = q_table[state, action]
        next_max = np.max(q_table[next_state])

        # Q-Learning Formel
        q_table[state, action] = (1 - alpha) * old_q + alpha * (reward + gamma * next_max)

        state = next_state

    # Epsilon langsam verringern
    epsilon = max(epsilon_min, epsilon * epsilon_decay)

# Abschluss-Check
if np.max(q_table) > 0:
    print("Das Training war erfolgreich! Die Q-Table ist gefüllt.")
    # Zeige die Werte für den Startzustand
    print("Empfohlene Aktionen für State 0 (Links, Unten, Rechts, Oben):")
    print(q_table[0])
else:
    print("WARNUNG: Immer noch alles 0. Versuche es erneut oder erhöhe epsilon_decay.")

In [ ]:
q_table

In [ ]:
import imageio
from IPython.display import Image, display

def create_robot_animation(q_table):
    # 1. Spezielles Environment für Grafik-Ausgabe erstellen (Folie 38)
    env_vis = gym.make("FrozenLake-v1", map_name="4x4", is_slippery=False, render_mode="rgb_array")
    env_vis = env

    state, info = env_vis.reset()
    frames = []
    terminated = False
    truncated = False

    # Den Startzustand aufnehmen
    frames.append(env_vis.render())

    print("Agent startet die autonome Navigation durch das Eisfeld...")

    while not (terminated or truncated):
        # Wähle die beste Aktion laut gelerntem Nutzen Q(s,a) (Folie 35)
        action = np.argmax(q_table[state])

        # Schritt ausführen (Interaktion mit der Umwelt - Folie 6)
        state, reward, terminated, truncated, info = env_vis.step(action)

        # Bild für die Animation speichern
        frames.append(env_vis.render())

    # Environment schließen
    env_vis.close()

    # Als GIF speichern (5 Bilder pro Sekunde)
    imageio.mimsave('frozen_lake_success.gif', frames, fps=5, loop=0)

    print("Animation fertiggestellt. Hier ist das Ergebnis:")
    display(Image(filename='frozen_lake_success.gif'))

# Demo mit deiner gefüllten Q-Table starten
create_robot_animation(q_table)

# 🛠 Laboraufgabe: Risiko vs. Sicherheit in der stochastischen Welt

### Ziel der Übung
In dieser Einheit untersuchen wir, wie ein Reinforcement Learning Agent mit Unsicherheit umgeht. Wir nutzen den Modus `is_slippery=True`, um unvorhersehbare Umwelteinflüsse zu simulieren.

---

### Aufgabe 1: Analyse der Robustheit (Theorie-Check)
Stellen Sie im Setup-Block `is_slippery=True` ein und trainieren Sie den Agenten erneut (min. 5000 Episoden). Starten Sie die Animation danach 5-mal hintereinander.

* **Beobachtung:** Warum erreicht der Agent fast immer das Ziel, obwohl er permanent in Richtungen rutscht, die er nicht gewählt hat?
* [cite_start]**Analyse:** Erklären Sie den Zusammenhang zwischen der **Transition $T(s, a \rightarrow s')$**  und der gelernten Sicherheit. Wie beeinflusst das Risiko eines "Rutschers" den Erwartungswert in der **Q-Table**?
* [cite_start]**Diskussion:** Warum ist der Agent hier ein Beispiel für **Non-Myopic Thinking**[cite: 149, 777]?

---

### Aufgabe 2: Das "Kurzsichtigkeits"-Experiment (Parameter-Tuning)
Untersuchen Sie den Einfluss des Diskontierungsfaktors **$\gamma$ (Gamma)**.
1. Setzen Sie `gamma = 0.1` (sehr klein) und trainieren Sie neu.
2. Beobachten Sie das Verhalten in der Animation.
* **Frage:** Wird der Agent risikofreudiger oder vorsichtiger?
* [cite_start]**Hintergrund:** Beziehen Sie sich auf das Problem der **Myopic Policy**. Warum sind Löcher plötzlich "weniger schlimm", wenn das Gamma klein ist?

---

### Aufgabe 3: Reward Shaping (Bonus für Schnelle)
[cite_start]Standardmäßig gibt es nur beim Erreichen des Ziels eine Belohnung ($r=1$)[cite: 5, 648].
* **Programmierung:** Ändern Sie den Trainings-Loop so, dass der Agent bei jedem Sturz in ein Loch eine **Bestrafung** erhält (z.B. `reward = -1`).
* **Auswertung:** Wie verändert sich die Anzahl der benötigten Trainings-Episoden, bis der Agent einen stabilen Pfad findet?
* [cite_start]**Transfer:** In welchen robotischen Szenarien (siehe Beispiele [cite: 13, 656]) ist eine Bestrafung für "totale Ausfälle" zwingend erforderlich?

